# Stage4:默认模式 vs 独立条件模式 —— 生图质量对比(n_cond=1)

**实验问题**:KV-Cache 的前提是「独立条件」注意力模式 —— 条件分支的 KV 在第 0 步写入后整段复用,
条件分支**看不到逐步演化的主图 latent**。这省了 ~1.5× 时间(stage3 已测),但质量掉不掉?

**对照设计**(控制变量,只动一个开关):

| | 默认模式 | 独立条件模式 |
|---|---|---|
| `kv_cache` | `False` | `True` |
| 条件分支 | 每步重算,参与联合注意力 | 第 0 步算一次,之后只读缓存 |
| LoRA / seed / prompt / steps | **完全相同**(自训 feature_reuse ckpt) | 同左 |

条件图 256px + `position_scale=2`(方案一),目标 512px,steps=28(FLUX.1-dev 质量档),canny 条件。

**结果呈现**:每个 case 一行三图(条件图 | 默认 | KV-Cache),盲评时 A/B 随机换边;
同时生成 `annotations.json` 骨架,人工标注 G/S/B 后用 `scripts/compute_gsb.py` 统计。

In [1]:
import os
os.chdir("..")  # 仓库根目录(与其它 repro notebook 一致)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import sys, glob, json, time, random
sys.path.insert(0, "."); sys.path.insert(0, "repro")

import torch
# 与 stage3/probe 相同的兜底:torch 2.8 + diffusers 0.38 的 VAE conv_in 问题
torch.backends.cudnn.enabled = False

from PIL import Image
from kvcache_benchmark import load_pipeline, attach_lora
from omini.pipeline.flux_omini import Condition, convert_to_condition, generate

TARGET_SIZE = 512
COND_SIZE   = 256
POS_SCALE   = TARGET_SIZE / COND_SIZE   # 位置编码网格必须拉伸对齐,否则错位
STEPS       = 28                        # dev 质量档(probe 用 3 步只是探显存)
SEEDS       = [42, 1234]
OUT_DIR     = "repro/quality_compare"
os.makedirs(OUT_DIR, exist_ok=True)

# WHY 不额外挂别的 LoRA:两种模式共用同一个自训 feature_reuse adapter,
# 它训练时 independent_condition=true,两种推理路径都合法 —— 这才是纯净对照。
hits = sorted(glob.glob("runs/*/ckpt/*/default.safetensors"), key=os.path.getmtime)
CKPT_DIR = os.environ.get("OMINI_CKPT_DIR") or os.path.dirname(hits[-1])
print(f"[DEBUG] setup: ckpt={CKPT_DIR} steps={STEPS} cond={COND_SIZE}px target={TARGET_SIZE}px")

11:45:28 [INFO] cudnn disabled (workaround for VAE CUDNN_STATUS_NOT_INITIALIZED)
11:45:30 [WARNING] Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.8.0+cu128).
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`


[DEBUG] setup: ckpt=runs/20260713-111742/ckpt/5000 steps=28 cond=256px target=512px


In [2]:
# 强制走本地 HF cache + 直接用 snapshot 路径(双保险,完全绕开网络)
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

LOCAL_FLUX_PATH = "/home/wuwenxuan03/.cache/huggingface/hub/models--black-forest-labs--FLUX.1-dev/snapshots/3de623fc3c33e44ffbe2bad470d0f45bccf2eb21"
print("HF_HUB_OFFLINE =", os.environ["HF_HUB_OFFLINE"])
print("LOCAL_FLUX_PATH exists:", os.path.isdir(LOCAL_FLUX_PATH))
print("LOCAL_FLUX_PATH has model_index.json:", os.path.isfile(os.path.join(LOCAL_FLUX_PATH, "model_index.json")))


HF_HUB_OFFLINE = 1
LOCAL_FLUX_PATH exists: True
LOCAL_FLUX_PATH has model_index.json: True


In [3]:
# 加载 pipeline(dispatch=auto → 本机 24G 卡走 2gpu:cuda:0=编码器, cuda:1=transformer 整块)
# 关键:直接传本地 snapshot 路径,绕开 HF Hub 远程调用,避免 gated repo 401
pipe = load_pipeline(LOCAL_FLUX_PATH, dispatch="auto")
attach_lora(pipe, CKPT_DIR, "default.safetensors", "feature_reuse")


11:45:36 [INFO] GPU 0: NVIDIA GeForce RTX 4090 | 23.6 GB
11:45:36 [INFO] GPU 1: NVIDIA GeForce RTX 4090 | 23.6 GB
11:45:36 [INFO] GPU 2: NVIDIA GeForce RTX 4090 | 23.6 GB
11:45:36 [INFO] GPU 3: NVIDIA GeForce RTX 4090 | 23.6 GB
11:45:36 [INFO] GPU 4: NVIDIA GeForce RTX 4090 | 23.6 GB
11:45:36 [INFO] GPU 5: NVIDIA GeForce RTX 4090 | 23.6 GB
11:45:36 [INFO] GPU 6: NVIDIA GeForce RTX 4090 | 23.6 GB
11:45:36 [INFO] GPU 7: NVIDIA GeForce RTX 4090 | 23.6 GB
11:45:36 [INFO] dispatch=auto -> 选择 2gpu(GPU0 24 GB)
11:45:36 [INFO] 加载 FLUX pipeline: /home/wuwenxuan03/.cache/huggingface/hub/models--black-forest-labs--FLUX.1-dev/snapshots/3de623fc3c33e44ffbe2bad470d0f45bccf2eb21 (bf16, 2gpu 手工 dispatch)


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

11:47:17 [INFO] dispatch 完成: cuda:0 = encoders+vae+latents(主场), cuda:1 = transformer
11:47:17 [INFO] 已安装 tx bridge:输入 -> transformer 卡,输出 -> 主场卡
11:47:17 [INFO] 加载 LoRA: runs/20260713-111742/ckpt/5000 :: default.safetensors
No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new
11:47:18 [INFO] device 安全网:搬了 0 个 param/buffer


FluxPipeline {
  "_class_name": "FluxPipeline",
  "_diffusers_version": "0.38.0",
  "_name_or_path": "/home/wuwenxuan03/.cache/huggingface/hub/models--black-forest-labs--FLUX.1-dev/snapshots/3de623fc3c33e44ffbe2bad470d0f45bccf2eb21",
  "feature_extractor": [
    null,
    null
  ],
  "image_encoder": [
    null,
    null
  ],
  "scheduler": [
    "diffusers",
    "FlowMatchEulerDiscreteScheduler"
  ],
  "text_encoder": [
    "transformers",
    "CLIPTextModel"
  ],
  "text_encoder_2": [
    "transformers",
    "T5EncoderModel"
  ],
  "tokenizer": [
    "transformers",
    "CLIPTokenizer"
  ],
  "tokenizer_2": [
    "transformers",
    "T5TokenizerFast"
  ],
  "transformer": [
    "diffusers",
    "FluxTransformer2DModel"
  ],
  "vae": [
    "diffusers",
    "AutoencoderKL"
  ]
}

In [4]:
# 测试 case:仓库自带 assets,挑结构线条清晰的(canny 条件才有信息量),
# 覆盖不同题材:静物 / 场景 / 生物 / 机械。prompt 与图内容对齐。
CASES = [
    ("vase",    "assets/vase_hq.jpg",     "A beautiful ceramic vase with flowers on a wooden table, soft light."),
    ("room",    "assets/room_corner.jpg", "A cozy room corner with an armchair and warm afternoon lighting."),
    ("oranges", "assets/oranges.jpg",     "Fresh ripe oranges on a plate, studio product photography."),
    ("clock",   "assets/clock.jpg",       "A vintage alarm clock on a desk, detailed, photorealistic."),
    ("rc_car",  "assets/rc_car.jpg",      "A red remote control toy car on the ground, sharp focus."),
]

def build_condition(image_path):
    img = Image.open(image_path).convert("RGB").resize((COND_SIZE, COND_SIZE))
    cond_img = convert_to_condition("canny", img)
    return cond_img, Condition(cond_img, "feature_reuse", position_scale=POS_SCALE)

def run_one(prompt, condition, seed, kv_cache):
    # WHY 每次重建 generator:两种模式必须从同一噪声出发,差异才全部归因于注意力模式
    g = torch.Generator(device="cuda:0").manual_seed(seed)
    t0 = time.perf_counter()
    out = generate(pipe, prompt=prompt, conditions=[condition],
                   height=TARGET_SIZE, width=TARGET_SIZE,
                   num_inference_steps=STEPS, guidance_scale=3.5,
                   generator=g, kv_cache=kv_cache)
    return out.images[0], time.perf_counter() - t0

In [5]:
# 主循环:5 case × 2 seed × 2 模式 = 20 次生成,steps=28 双卡下预计 20~30 分钟
records = []
for name, path, prompt in CASES:
    cond_img, condition = build_condition(path)
    cond_img.save(f"{OUT_DIR}/{name}_cond.png")
    for seed in SEEDS:
        row = {"case": name, "seed": seed, "prompt": prompt}
        for mode, kv in (("default", False), ("kvcache", True)):
            img, dt = run_one(prompt, condition, seed, kv)
            fp = f"{OUT_DIR}/{name}_s{seed}_{mode}.png"
            img.save(fp)
            row[mode] = fp; row[f"{mode}_sec"] = round(dt, 1)
        records.append(row)
        print(f"[DEBUG] run_one: {name} seed={seed} "
              f"default={row['default_sec']}s kvcache={row['kvcache_sec']}s "
              f"speedup={row['default_sec']/row['kvcache_sec']:.2f}x")

with open(f"{OUT_DIR}/records.json", "w") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)
print(f"完成 {len(records)} 组对照,已写 {OUT_DIR}/records.json")

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

[DEBUG] run_one: vase seed=42 default=8.0s kvcache=5.8s speedup=1.38x


  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

...

  0%|          | 0/28 [00:00<?, ?it/s]

[DEBUG] run_one: rc_car seed=42 default=7.4s kvcache=5.8s speedup=1.28x


  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

[DEBUG] run_one: rc_car seed=1234 default=7.4s kvcache=5.8s speedup=1.28x
完成 10 组对照,已写 repro/quality_compare/records.json


In [ ]:
# 可视化:调独立脚本(自洽,不依赖 kernel 状态/OUT_DIR 变量,任何 cwd 下都能跑)
# 脚本: repro/visualize_quality_compare.py
# 产物: repro/quality_compare/comparison_grid.png
!python repro/visualize_quality_compare.py


In [ ]:
# 生成 GSB 标注骨架(格式对齐 scripts/compute_gsb.py 的读取方式)。
# WHY 随机换边:盲评时不知道哪边是 kv_cache,避免先入为主。side_map 单独存,统计时再翻译。
rng = random.Random(0)
annotations, side_map = {}, {}
for r in records:
    cid = f"{r['case']}_s{r['seed']}"
    flip = rng.random() < 0.5
    side_map[cid] = {"A": "kvcache" if flip else "default",
                     "B": "default" if flip else "kvcache"}
    # choices 留空待人工填:G = A 好(kv 相对基线的口径见 side_map), S = 相当, B = A 差
    annotations[cid] = {"prompt": r["prompt"], "choices": {"quality": ""}}

with open(f"{OUT_DIR}/annotations.json", "w") as f:
    json.dump({"annotations": annotations}, f, ensure_ascii=False, indent=2)
with open(f"{OUT_DIR}/side_map.json", "w") as f:
    json.dump(side_map, f, ensure_ascii=False, indent=2)
print("标注流程:肉眼对比每组两图 → 在 annotations.json 的 choices.quality 填 G/S/B")
print("(以 kv_cache 相对 default 的口径:G=更好 S=相当 B=更差,填完跑下面这条)")
print(f"  python scripts/compute_gsb.py {OUT_DIR}/annotations.json")

## 小结

- 20 次生成(5 case × 2 seed × 2 模式),唯一变量是 `kv_cache` 开关,同 LoRA 同 seed 同噪声。
- 三个产物:`records.json`(路径+耗时)、成对 PNG(可视化网格)、`annotations.json` + `side_map.json`(GSB 盲评)。
- 附带副产物:`default_sec / kvcache_sec` 即 n_cond=1、steps=28 下的真实加速比,可与 stage3 的 1.5~1.6× 互相印证。
- 预期:若两列图肉眼几乎不可分(GSB ≈ 1),说明独立条件模式在 n_cond=1 下质量无损,可放心用 KV-Cache。
- 可视化已抽到独立脚本 `repro/visualize_quality_compare.py`,kernel 重启后单独跑 `python repro/visualize_quality_compare.py` 也能重出图,不依赖 cell 5 的内存状态。
